## Prerequisite Code

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ../initial-setup/03-utils

In [0]:
# View loaded utils
print(f'Warrior Schemas: {wr_bronze_schema}, {wr_silver_schema}, {wr_gold_schema}')

print(f'Sports Direct Schema: {sd_gold_schema}')

Warrior Schemas: warrior_bronze, warrior_silver, warrior_gold
Sports Direct Schema: sportsdirect_gold


In [0]:
# Create widgets
dbutils.widgets.text('catalog', 'sportsdirect_sales', 'Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')

In [0]:
# Access widgets
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

# View values
print(f'Catalog: {catalog}')
print(f'Data Source: {data_source}')

Catalog: sportsdirect_sales
Data Source: products


In [0]:
# Define source directory
source_dir = f's3://sd-warrior-acquisition/{data_source}/*.csv'

# View source directory
# dbutils.fs.ls(source_dir)

## Warrior Bronze Layer

In [0]:
# Read data from the source directory
raw_data = spark.read \
    .format('csv') \
    .option('header', True) \
    .option('inferSchema', True) \
    .load(source_dir) \
    .withColumn('read_timestamp', F.current_timestamp()) \
    .select('*', '_metadata.file_name', '_metadata.file_size')

# View raw data
display(raw_data)

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-03-17T12:24:30.020Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-03-17T12:24:30.020Z,products.csv,1388


In [0]:
# Write raw data to the bronze table
raw_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', True) \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_bronze_schema}.dim_product')

## Warrior Silver Layer

In [0]:
# Clean data and apply tranformations

# Fix typos in the product_name column
transformed_data = raw_data. \
    withColumn(
        'product_name',
        F.regexp_replace(
            F.col('product_name'),
            r'(?i)protien',
            'Protein'
        )
    )

# Drop duplicate products
transformed_data = transformed_data.dropDuplicates(['product_name', 'product_id', 'category'])

# Extract variant from product_name
transformed_data = transformed_data. \
    withColumn(
        'variant',
        F.regexp_extract(
            F.col('product_name'),
            r'\(([^)]+)\)',
            1
        )
    )

# Remove variant from product_name
transformed_data = transformed_data. \
    withColumn(
        'product_name',
        F.regexp_replace(
            F.col('product_name'),
            r'\s*\([^)]+\)',
            ''
        )
    )

# Fix typos in the category column
transformed_data = transformed_data. \
    withColumn(
        'category',
        F.regexp_replace(
            F.col('category'),
            r'(?i)protien',
            'protein'
        )
    )

# Fix category casing
transformed_data = transformed_data. \
    withColumn(
        'category',
        F.initcap(F.col('category'))
    )

# Add a division column
transformed_data = transformed_data. \
    withColumn(
        'division',
        F.when(F.col('category').isin(['Energy Bars', 'Protein Bars']), 'Nutrition Bars')
            .when(F.col('category') == 'Granola & Cereals', 'Breakfast Foods')
            .when(F.col('category') == 'Recovery Dairy', 'Dairy & Recovery')
            .when(F.col('category') == 'Healthy Snacks', 'Healthy Snacks')
            .when(F.col('category') == 'Electrolyte Mix', 'Hydration & Electrolytes')
            .otherwise('Other')
    )

# Replace invalid product_ids with 999999
transformed_data = transformed_data. \
    withColumn(
        'product_id',
        F.when(F.col('product_id').rlike('^[0-9]+$'), F.col('product_id'))
            .otherwise(F.lit('999999').cast('string'))
    )

# Add a product_code column
transformed_data = transformed_data. \
    withColumn(
        'product_code',
        F.sha2(F.concat(F.col('product_name'), F.col('product_id')), 256)
    )

# Rename the product_name column
transformed_data = transformed_data. \
    withColumnRenamed(
        'product_name',
        'product'
    )

In [0]:
# Verify typo fix
display(transformed_data.filter(F.col('product_name').contains('protien')).count())

0

In [0]:
# Check for no duplicate products
# display(transformed_data.groupBy('product_name').count().filter(F.col('count') > 1))

In [0]:
# Check variant
# display(transformed_data.select('product_name', 'variant').orderBy('product_name'))

In [0]:
# Verify typo fix
display(transformed_data.filter(F.col('category').contains('protien')).count())

0

In [0]:
# Check category casing
display(transformed_data.select('category').distinct())

category
Protein Bars
Healthy Snacks
Energy Bars
Recovery Dairy
Granola & Cereals
Electrolyte Mix


In [0]:
# Verify divisions
display(transformed_data.select('category', 'division').distinct())

category,division
Healthy Snacks,Healthy Snacks
Granola & Cereals,Breakfast Foods
Recovery Dairy,Dairy & Recovery
Energy Bars,Nutrition Bars
Electrolyte Mix,Hydration & Electrolytes
Protein Bars,Nutrition Bars


In [0]:
# Verify product_ids and product_codes
display(transformed_data)

product,product_id,category,read_timestamp,file_name,file_size,variant,division,product_code
SportsBar Oats Cookie Bites ChocoChip,999999,Healthy Snacks,2026-03-17T12:52:24.918Z,products.csv,1388,350g,Healthy Snacks,6ccf418552ffec70fde2e99b77ebbd1220e2d091c3d3cd5e34fd59f868b761c2
SportsBar Granola Crunch Honey Almond,25891302,Granola & Cereals,2026-03-17T12:52:24.918Z,products.csv,1388,300g,Breakfast Foods,81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c
SportsBar Granola Crunch Honey Almond,25891303,Granola & Cereals,2026-03-17T12:52:24.918Z,products.csv,1388,200g,Breakfast Foods,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5
SportsBar Oats Cookie Bites ChocoChip,25891503,Healthy Snacks,2026-03-17T12:52:24.918Z,products.csv,1388,180g,Healthy Snacks,c3a270caf0285f44bbd624e9a209a46e86f47adf81d2bcadf5969ff260d97d12
SportsBar Greek Yogurt Pro Vanilla,25891403,Recovery Dairy,2026-03-17T12:52:24.918Z,products.csv,1388,80g,Dairy & Recovery,8395d734ede7e81a35c67ea3ae7240db0380f8370fb488a3872ea033af705ee4
SportsBar Greek Yogurt Pro Vanilla,25891401,Recovery Dairy,2026-03-17T12:52:24.918Z,products.csv,1388,200g,Dairy & Recovery,958f41cd549041a991f52c5080db909845931526051f6a535e0221223259b5e2
SportsBar Energy Bar Choco Fudge,25891101,Energy Bars,2026-03-17T12:52:24.918Z,products.csv,1388,60g,Nutrition Bars,95dd546ad1c0e319431aabb5d05da6af9d0418dbf7decda178337b8c26cc898f
SportsBar Energy Bar Choco Fudge,25891102,Energy Bars,2026-03-17T12:52:24.918Z,products.csv,1388,40g,Nutrition Bars,00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f
SportsBar Electrolyte Mix Lemon-Lime,25891601,Electrolyte Mix,2026-03-17T12:52:24.918Z,products.csv,1388,30 Sachets,Hydration & Electrolytes,19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb
SportsBar Greek Yogurt Pro Vanilla,25891402,Recovery Dairy,2026-03-17T12:52:24.918Z,products.csv,1388,120g,Dairy & Recovery,41d9f8038c4771bf55fdddeaf9e940f5e23b717d63c6a992e2654afc37fc2c8d


In [0]:
# Verify renamed column
display(transformed_data)

product,product_id,category,read_timestamp,file_name,file_size,variant,division,product_code
SportsBar Oats Cookie Bites ChocoChip,999999,Healthy Snacks,2026-03-17T12:54:13.251Z,products.csv,1388,350g,Healthy Snacks,6ccf418552ffec70fde2e99b77ebbd1220e2d091c3d3cd5e34fd59f868b761c2
SportsBar Granola Crunch Honey Almond,25891302,Granola & Cereals,2026-03-17T12:54:13.251Z,products.csv,1388,300g,Breakfast Foods,81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c
SportsBar Granola Crunch Honey Almond,25891303,Granola & Cereals,2026-03-17T12:54:13.251Z,products.csv,1388,200g,Breakfast Foods,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5
SportsBar Oats Cookie Bites ChocoChip,25891503,Healthy Snacks,2026-03-17T12:54:13.251Z,products.csv,1388,180g,Healthy Snacks,c3a270caf0285f44bbd624e9a209a46e86f47adf81d2bcadf5969ff260d97d12
SportsBar Greek Yogurt Pro Vanilla,25891403,Recovery Dairy,2026-03-17T12:54:13.251Z,products.csv,1388,80g,Dairy & Recovery,8395d734ede7e81a35c67ea3ae7240db0380f8370fb488a3872ea033af705ee4
SportsBar Greek Yogurt Pro Vanilla,25891401,Recovery Dairy,2026-03-17T12:54:13.251Z,products.csv,1388,200g,Dairy & Recovery,958f41cd549041a991f52c5080db909845931526051f6a535e0221223259b5e2
SportsBar Energy Bar Choco Fudge,25891101,Energy Bars,2026-03-17T12:54:13.251Z,products.csv,1388,60g,Nutrition Bars,95dd546ad1c0e319431aabb5d05da6af9d0418dbf7decda178337b8c26cc898f
SportsBar Energy Bar Choco Fudge,25891102,Energy Bars,2026-03-17T12:54:13.251Z,products.csv,1388,40g,Nutrition Bars,00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f
SportsBar Electrolyte Mix Lemon-Lime,25891601,Electrolyte Mix,2026-03-17T12:54:13.251Z,products.csv,1388,30 Sachets,Hydration & Electrolytes,19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb
SportsBar Greek Yogurt Pro Vanilla,25891402,Recovery Dairy,2026-03-17T12:54:13.251Z,products.csv,1388,120g,Dairy & Recovery,41d9f8038c4771bf55fdddeaf9e940f5e23b717d63c6a992e2654afc37fc2c8d


In [0]:
# Write data to the silver table
transformed_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', 'true') \
    .option('mergeSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_silver_schema}.dim_product')

## Warrior Gold Layer

In [0]:
# Get analytics data from the silver layer
analytics_data = transformed_data.select('product_code', 'division', 'category', 'product', 'variant')

In [0]:
# Write analytics data to the gold table
analytics_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_gold_schema}.dim_product')

## Sports Direct Gold Layer

In [0]:
# Merge analytics data from the Warrior gold layer to the Sports Direct gold layer
sd_dim_product_table = DeltaTable.forName(spark, f'{catalog}.{sd_gold_schema}.dim_product')

sd_dim_product_table.alias('target').merge(
    source=analytics_data.alias('source'),
    condition='target.product_code = source.product_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]